# Exchange-Traded Bond Options in LUSID

| Section | Topic |
|---|---|
| 1 | Instrument creation |
| 2 | Recipe |
| 3 | Transaction types |
| 4 | Portfolio and transactions |
| 5 | Valuation |
| 6 | Instrument events |

## The instrument

An exchange-traded bond option is an option on a bond **future**, not on a cash bond. The
contracts that actually trade on-exchange -- CBOT-style 10Y note options, for example -- are
American-exercise, physically-settled options that deliver the underlying future when exercised.
An option directly on a cash bond, rather than on a future, is a different LUSID type --
`BondOption` -- not covered here.

LUSID models this as an `ExchangeTradedOption`. Its `contract_details.underlying` field takes any
inline `LusidInstrument`, and here that's a `Future`, built inline rather than mastered as its own
instrument first.

---
## Setup

In [1]:
import os
import json
import certifi
os.environ.setdefault("SSL_CERT_FILE", certifi.where())

from datetime import datetime, timezone, timedelta
import pandas as pd

import lusid
import lusid.models as m
from lusid.extensions import (
    SyncApiClientFactory, SecretsFileConfigurationLoader, EnvironmentVariablesConfigurationLoader)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.options.display.float_format = "{:,.2f}".format

# Looks for a secrets file at FBN_SECRETS_PATH, then secrets.json in this folder; falls back to
# LUSID's own environment-variable config (FBN_LUSID_URL, FBN_TOKEN_URL, FBN_USERNAME,
# FBN_PASSWORD, FBN_CLIENT_ID, FBN_CLIENT_SECRET, FBN_APP_NAME) if neither is present. See
# secrets.example.json for the file's shape.
SECRETS_PATH = os.getenv("FBN_SECRETS_PATH") or (
    "secrets.json" if os.path.exists("secrets.json") else None)
config_loaders = ([SecretsFileConfigurationLoader(SECRETS_PATH)] if SECRETS_PATH
                   else [EnvironmentVariablesConfigurationLoader()])

factory = SyncApiClientFactory(config_loaders=config_loaders)


def api(cls):
    return factory.build(cls)


instruments_api   = api(lusid.InstrumentsApi)
txn_portfolio_api = api(lusid.TransactionPortfoliosApi)
portfolios_api    = api(lusid.PortfoliosApi)
quotes_api        = api(lusid.QuotesApi)
recipes_api       = api(lusid.ConfigurationRecipeApi)
aggregation_api   = api(lusid.AggregationApi)

meta = api(lusid.ApplicationMetadataApi).get_lusid_versions()
href = meta.links[0].href
print("Domain      :", href[:href.find("/app/")] if "/app/" in href else href)
print("API version :", meta.build_version)

Domain      : https://fbn-tejan.lusid.com
API version : 0.6.16597.0


Two more API clients get used from section 3 onwards: `TransactionConfigurationApi` for
registering transaction types, and `InstrumentEventTypesApi` plus `InstrumentEventsApi` for
inspecting and forecasting instrument events.

In [2]:
txn_config_api  = api(lusid.TransactionConfigurationApi)
events_api      = api(lusid.InstrumentEventsApi)
event_types_api = api(lusid.InstrumentEventTypesApi)

---
## Configuration

The option premium is quoted in points on the future's contract size (1 point = 1% of
`CONTRACT_SIZE`) -- a common quoting convention for this kind of contract. `SimpleStatic` just
reports the position at that quoted mark, so PV works out to contracts held x (quote / 100) x
contract size.

In [3]:
def d(year, month, day):
    return datetime(year, month, day, tzinfo=timezone.utc)


def upsert(key, name, client_internal, definition):
    """Upsert one instrument and return its LUID."""
    resp = instruments_api.upsert_instruments(scope=SCOPE, request_body={
        key: m.InstrumentDefinition(
            name=name,
            identifiers={"ClientInternal": m.InstrumentIdValue(value=client_internal)},
            definition=definition)})
    assert not resp.failed, list(resp.failed.values())[0].detail
    return resp.values[key].lusid_instrument_id


def mastered(luid):
    """Reference an instrument that already exists in the master."""
    return m.MasteredInstrument(
        instrument_type="MasteredInstrument",
        identifiers={"Instrument/default/LusidInstrumentId": luid})


def recreate_portfolio(code, display_name, base_currency, created, recipe=None):
    """Create the portfolio, replacing any earlier run so the book starts empty."""
    request = m.CreateTransactionPortfolioRequest(
        display_name=display_name, code=code, base_currency=base_currency,
        created=created, instrument_scopes=[SCOPE],
        instrument_event_configuration=None if recipe is None else
        m.InstrumentEventConfiguration(
            transaction_template_scopes=["default"],
            recipe_id=m.ResourceId(scope=SCOPE, code=recipe)))
    try:
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Created {SCOPE}/{code}")
    except lusid.ApiException as e:
        if "PortfolioWithIdAlreadyExists" not in str(getattr(e, "body", "")):
            raise
        portfolios_api.delete_portfolio(scope=SCOPE, code=code)
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Recreated {SCOPE}/{code}")


def upsert_price(luid, price, effective, currency):
    """One Price/mid quote, keyed on the instrument's LUID."""
    quotes_api.upsert_quotes(scope=SCOPE, request_body={
        f"{luid}-{effective:%Y%m%d}": m.UpsertQuoteRequest(
            quote_id=m.QuoteId(
                quote_series_id=m.QuoteSeriesId(
                    provider="Lusid", instrument_id=luid,
                    instrument_id_type="LusidInstrumentId",
                    quote_type="Price", field="mid"),
                effective_at=effective.isoformat()),
            metric_value=m.MetricValue(value=price, unit=currency))})


def value(portfolio, effective, metrics, currency, group_by=None):
    """Run the recipe over one portfolio and return the result as a DataFrame."""
    request = m.ValuationRequest(
        recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE),
        metrics=[m.AggregateSpec(key=k, op=op) for k, op in metrics],
        group_by=group_by or ["Instrument/default/Name"],
        report_currency=currency,
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=portfolio, portfolio_entity_type="SinglePortfolio")],
        valuation_schedule=m.ValuationSchedule(effective_at=effective.isoformat()))
    return pd.DataFrame(aggregation_api.get_valuation(valuation_request=request).data)


def transactions(portfolio, from_date, as_at):
    """The portfolio's own booked transactions over a date range, as a DataFrame."""
    txns = txn_portfolio_api.get_transactions(
        scope=SCOPE, code=portfolio,
        from_transaction_date=from_date.isoformat(),
        to_transaction_date=as_at.isoformat()).values
    if not txns:
        return pd.DataFrame(columns=["date", "type", "luid", "units", "consideration"])
    return pd.DataFrame([{
        "date": pd.Timestamp(t.transaction_date).strftime("%Y-%m-%d"),
        "type": t.type,
        "luid": t.instrument_uid,
        "units": t.units,
        "consideration": t.total_consideration.amount,
    } for t in txns]).sort_values(["date", "type"]).reset_index(drop=True)


SCOPE     = "BondOptionDemo"
RECIPE    = "bond-option-demo-recipe"
PORTFOLIO = "bond-option-demo-book"

FUTURE_ID   = "DEMO-10YFUT-Z25"
FUTURE_DESC = "Demo 10Y Note Future Dec-25"

OPTION_ID   = "DEMO-10YFUT-OPT-CALL-Z25"
DESC        = "Demo 10Y Note Future Dec-25 112.50 Call (American, Physical)"

CURRENCY      = "USD"
STRIKE        = 112.50
OPTION_TYPE   = "Call"
EXERCISE_TYPE = "American"
DELIVERY_TYPE = "Physical"

FUTURE_START    = d(2025, 1, 2)
FUTURE_MATURITY = d(2025, 12, 19)   # future's delivery date

TRADE_DATE = d(2025, 3, 3)          # option start date / transaction date
EXPIRY     = d(2025, 11, 21)        # exercise date -- before the future's maturity
ASOF       = d(2025, 9, 15)         # valuation date -- before expiry

CONTRACT_SIZE  = 100_000.0          # face value per contract; 1 point = 1,000
EXCHANGE_CODE  = "DEMO-CBT"
COUNTRY        = "US"
ASSET_CLASS    = "InterestRates"
CONTRACT_CODE  = "D10"
CONTRACT_MONTH = "Z"                # December

QUANTITY = 25.0                     # contracts held
PRICE    = 2.50                     # points, quoted mark
DENOM    = 100

print(f"{DESC}")
print(f"  underlying {FUTURE_DESC} ({FUTURE_ID})")
print(f"  strike {STRIKE}, {EXERCISE_TYPE} exercise, {DELIVERY_TYPE} settlement, expiry {EXPIRY:%Y-%m-%d}")
print(f"  {QUANTITY:,.0f} contracts at {PRICE} points on {ASOF:%Y-%m-%d}")
print(f"  market value = {QUANTITY:,.0f} x {PRICE}/{DENOM} x {CONTRACT_SIZE:,.0f} "
      f"= {QUANTITY * PRICE / DENOM * CONTRACT_SIZE:,.2f} {CURRENCY}")

Demo 10Y Note Future Dec-25 112.50 Call (American, Physical)
  underlying Demo 10Y Note Future Dec-25 (DEMO-10YFUT-Z25)
  strike 112.5, American exercise, Physical settlement, expiry 2025-11-21
  25 contracts at 2.5 points on 2025-09-15
  market value = 25 x 2.5/100 x 100,000 = 62,500.00 USD


---
# 1. Instrument creation

The future is built inline as the option's `contract_details.underlying`. `ExchangeTradedOption`
will accept any `LusidInstrument` there, which is why the future doesn't need to be mastered
separately first.

In [4]:
future = m.Future(
    instrument_type="Future",
    start_date=FUTURE_START,
    maturity_date=FUTURE_MATURITY,
    identifiers={"ClientInternal": FUTURE_ID},
    contract_details=m.FuturesContractDetails(
        dom_ccy=CURRENCY,
        asset_class=ASSET_CLASS,
        contract_code=CONTRACT_CODE,
        contract_month=CONTRACT_MONTH,
        contract_size=CONTRACT_SIZE,
        country=COUNTRY,
        description=FUTURE_DESC,
        exchange_code=EXCHANGE_CODE,
        delivery_type="Physical"),
    contracts=1,
    ref_spot_price=0.0)

option = m.ExchangeTradedOption(
    instrument_type="ExchangeTradedOption",
    start_date=TRADE_DATE,
    contracts=1,
    ref_spot_price=STRIKE,
    contract_details=m.ExchangeTradedOptionContractDetails(
        dom_ccy=CURRENCY,
        strike=STRIKE,
        contract_size=CONTRACT_SIZE,
        country=COUNTRY,
        delivery_type=DELIVERY_TYPE,
        description=DESC,
        exchange_code=EXCHANGE_CODE,
        exercise_date=EXPIRY,
        exercise_type=EXERCISE_TYPE,
        option_code=OPTION_ID,
        option_type=OPTION_TYPE,
        underlying=future,
        underlying_code=FUTURE_ID))

OPTION_LUID = upsert("option", DESC, OPTION_ID, option)
print(f"Bond option : {OPTION_LUID}")

Bond option : LUID_00003DFY


---
# 2. Recipe

With `SimpleStatic`, the position gets reported at a quoted mark, so the contract details on the
instrument only need to describe what it *is* -- under this model, they don't feed into the
valuation number itself.

In [5]:
recipes_api.upsert_configuration_recipe(
    upsert_recipe_request=m.UpsertRecipeRequest(
        configuration_recipe=m.ConfigurationRecipe(
            scope=SCOPE, code=RECIPE,
            description="Exchange-traded bond option, marked",
            market=m.MarketContext(
                market_rules=[m.MarketDataKeyRule(
                    key="Quote.LusidInstrumentId.*", supplier="Lusid", data_scope=SCOPE,
                    quote_type="Price", field="mid", quote_interval="1Y")],
                options=m.MarketOptions(
                    default_supplier="Lusid",
                    default_instrument_code_type="LusidInstrumentId",
                    default_scope=SCOPE)),
            pricing=m.PricingContext(
                model_rules=[m.VendorModelRule(
                    supplier="Lusid", model_name="SimpleStatic",
                    instrument_type="ExchangeTradedOption")],
                options=m.PricingOptions(allow_partially_successful_evaluation=True)))))

print(f"Recipe: {SCOPE}/{RECIPE}")

Recipe: BondOptionDemo/bond-option-demo-recipe


---
# 3. Transaction types

`ExpiryEvent` and `OptionExercisePhysicalEvent`/`OptionExerciseCashEvent` are the events LUSID
derives from an `ExchangeTradedOption`'s own `exercise_date` and `delivery_type`. But
`query_applicable_instrument_events` will return them with an **empty** `.transactions` list until
a transaction type exists for the template to emit -- and the transaction types to register are
`Expiry` and `OptionExercisePhysical`.

In [6]:
for event_type in ("ExpiryEvent", "OptionExercisePhysicalEvent"):
    spec = event_types_api.get_transaction_template_specification(instrument_event_type=event_type)
    print(event_type)
    print(f"  applies to    : {spec.supported_instrument_types}")
    print(f"  participation : {spec.supported_participation_types}")
    print(f"  elections     : {[e.election_type for e in (spec.supported_election_types or [])]}")
    print()

ExpiryEvent
  applies to    : ['InterestRateSwaption', 'EquityOption', 'ExchangeTradedOption', 'ToBeAnnouncedOption', 'BondOption', 'CdsOption']
  participation : ['Mandatory']
  elections     : []



OptionExercisePhysicalEvent
  applies to    : ['InterestRateSwaption', 'EquityOption', 'ExchangeTradedOption', 'ToBeAnnouncedOption', 'CdsOption']
  participation : ['Voluntary']
  elections     : ['OptionExerciseElection']



`ExpiryEvent` maps to the transaction type `Expiry` -- just the bare event name, not an
instrument-qualified one.

`OptionExercisePhysicalEvent`'s `supportedParticipationTypes` comes back as `Voluntary` (more on
what that means in section 6). We register its transaction type below too, so it's ready in case a
future election needs it.

In [7]:
TXN_TYPES = [
    ("Expiry",                    "Option expires unexercised", -1),
    ("OptionExercisePhysical",    "Option exercised, physical delivery of the future", -1),
]

for txn_type, description, direction in TXN_TYPES:
    txn_config_api.set_transaction_type(
        source="default", type=txn_type, scope="default",
        transaction_type_request=m.TransactionTypeRequest(
            aliases=[m.TransactionTypeAlias(
                type=txn_type, description=description,
                transaction_class="Basic", transaction_roles="AllRoles", is_default=False)],
            movements=[m.TransactionTypeMovement(
                movement_types="StockMovement", side="Side1", direction=direction)]))
    print(f"{txn_type:<24} StockMovement Side1 {direction:+d}")

Expiry                   StockMovement Side1 -1


OptionExercisePhysical   StockMovement Side1 -1


---
# 4. Portfolio and transactions

`recreate_portfolio(..., recipe=RECIPE)` sets `instrumentEventConfiguration` at creation time,
which is the only time it can be set. That's what points this book's event forecasting at the
recipe from section 2 -- skip it, and section 6 will quietly return zero events instead of
raising an error.

The trade itself is booked as a `Buy` of 25 contracts at the same points price used for the mark,
so `totalConsideration` works out to the premium paid.

In [8]:
recreate_portfolio(PORTFOLIO, "Bond Option Demo Book", CURRENCY, d(2025, 1, 1), recipe=RECIPE)

PREMIUM = QUANTITY * PRICE / DENOM * CONTRACT_SIZE

txn_portfolio_api.upsert_transactions(
    scope=SCOPE, code=PORTFOLIO,
    transaction_request=[m.TransactionRequest(
        transaction_id="BUY-OPT",
        type="Buy",
        instrument_identifiers={"Instrument/default/LusidInstrumentId": OPTION_LUID},
        transaction_date=TRADE_DATE.isoformat(),
        settlement_date=TRADE_DATE.isoformat(),
        units=QUANTITY,
        transaction_price=m.TransactionPrice(price=PRICE / DENOM, type="Price"),
        total_consideration=m.CurrencyAndAmount(amount=PREMIUM, currency=CURRENCY),
        source="default")])

display(transactions(PORTFOLIO, TRADE_DATE, TRADE_DATE))

Recreated BondOptionDemo/bond-option-demo-book


,date,type,luid,units,consideration
0,2025-03-03,Buy,LUID_00003DFY,25.00,"62,500.00"


---
# 5. Valuation

Just one quote needed here, at the option's own points price.

In [9]:
upsert_price(OPTION_LUID, PRICE / DENOM, ASOF, CURRENCY)

METRICS = [("Instrument/default/Name", "Value"),
           ("Holding/default/Units",   "Sum"),
           ("Valuation/PV",            "Sum")]

result = value(PORTFOLIO, ASOF, METRICS, CURRENCY)
display(result)

pv = result.loc[result["Instrument/default/Name"] == DESC, "Sum(Valuation/PV)"].iloc[0]
expected = QUANTITY * PRICE / DENOM * CONTRACT_SIZE
print(f"LUSID PV {pv:,.2f}  vs  quoted mark {expected:,.2f}")

,Instrument/default/Name,Sum(Holding/default/Units),Sum(Valuation/PV)
0,Demo 10Y Note Future Dec-25 112.50 Call (Ameri...,25.00,"62,500.00"
1,USD,"-62,500.00","-62,500.00"


LUSID PV 62,500.00  vs  quoted mark 62,500.00


---
# 6. Instrument events

## 6a. Applicable events

`query_applicable_instrument_events` forecasts every event this book's holding implies over a
window spanning the option's own `exercise_date`. We'd expect one: `ExpiryEvent`, `Mandatory`
participation -- the default outcome when nothing's been elected.

`OptionExercisePhysicalEvent`, the event this `Physical`-delivery option would carry if it were
exercised, does **not** show up here, and registering a transaction type for it in section 3
doesn't change that. Its `supportedParticipationTypes` came back `Voluntary`, and LUSID only
forecasts a voluntary outcome once an election instruction exists for the holding -- which in turn
needs a corporate action source set up on the portfolio. This notebook doesn't add that piece, so
`ExpiryEvent` is the only outcome it can forecast.

In [10]:
WINDOW_END = EXPIRY + timedelta(days=10)

applicable = events_api.query_applicable_instrument_events(
    query_applicable_instrument_events_request=m.QueryApplicableInstrumentEventsRequest(
        window_start=TRADE_DATE.isoformat(),
        window_end=WINDOW_END.isoformat(),
        effective_at=WINDOW_END.isoformat(),
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=PORTFOLIO, portfolio_entity_type="SinglePortfolio")],
        forecasting_recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE))).values

print(f"{len(applicable)} applicable event(s):\n")
display(pd.DataFrame([{
    "event type": ev.instrument_event_type,
    "eligible balance": ev.eligible_balance,
    "status": ev.instrument_event_status,
} for ev in applicable]))

1 applicable event(s):



,event type,eligible balance,status
0,ExpiryEvent,25.00,Active


## 6b. The transactions the event carries

`ExpiryEvent` now carries one transaction of type `Expiry`, the type registered in section 3,
because that type now exists in the `default` scope. Before we registered it, this list came back
empty.

In [11]:
rows = []
for ev in applicable:
    for txn in (ev.transactions or []):
        rows.append({
            "event": ev.instrument_event_type,
            "txn type": getattr(txn, "type", None),
            "units": getattr(txn, "units", None),
            "price": getattr(getattr(txn, "transaction_price", None), "price", None),
        })

if rows:
    display(pd.DataFrame(rows))
else:
    print("No forecast transactions. Check the transaction types in section 3.")

,event,txn type,units,price
0,ExpiryEvent,Expiry,25.00,0.00


---
# Summary

1. An American, physically-settled option on a bond future is an `ExchangeTradedOption`, with the
   `Future` carried inline in `contract_details.underlying`.
2. That underlying field takes any `LusidInstrument`, so the `Future` never needs its own
   mastering step -- it's built right there inline.
3. `SimpleStatic` reports the position at a quoted mark, so PV comes out to contracts held x
   (quote / 100) x `contract_size`.
4. `ExpiryEvent`'s and `OptionExercise*Event`'s `.transactions` stay empty until a matching
   transaction type gets registered: `Expiry` and `OptionExercisePhysical`, the bare event names.
5. Only `ExpiryEvent` actually populated in this notebook. `OptionExercisePhysicalEvent` never
   shows up as an applicable event here, no matter what transaction type is registered for it,
   because it has `Voluntary` participation -- LUSID only forecasts a voluntary outcome once an
   election exists for the holding, and that needs a corporate action source this notebook doesn't
   set up. So the expiry path is shown end to end, while the exercise path isn't.

In [12]:
print(f"Scope      : {SCOPE}")
print(f"Portfolio  : {SCOPE}/{PORTFOLIO}")
print(f"Recipe     : {SCOPE}/{RECIPE}")
print(f"Instrument : {OPTION_LUID}")

Scope      : BondOptionDemo
Portfolio  : BondOptionDemo/bond-option-demo-book
Recipe     : BondOptionDemo/bond-option-demo-recipe
Instrument : LUID_00003DFY
